# The Binomial Distribution

This notebook introduces the first named distribution of the course: the **binomial distribution**, which models the number of successes (heads, ones, defects, ...) in a fixed number of independent trials. Along the way it answers the last notebook's open question — how to *count* orderings without listing them — with the **binomial coefficient**. It is based on the third video of Lesson 2 (Probability Distributions).


## 1. Introduction

Toss a coin 10 times: the number of heads can be $0, 1, \dots, 10$, each with its own probability. The histogram of those probabilities *is* the binomial distribution — the simplest example of a **discrete distribution** (continuous ones come later in the course).

After completing this notebook you will be able to:

- Count orderings with the binomial coefficient $\binom{n}{k}$.
- Write the binomial PMF and the notation $X \sim \text{Binomial}(n, p)$.
- Recognize binomial experiments in disguise (a die as a biased coin).


## 2. Theory

### Counting orderings: the binomial coefficient

What is the probability of exactly 2 heads in 5 flips? One specific sequence like $HHTTT$ has probability $\left(\frac{1}{2}\right)^5 = \frac{1}{32}$ — and *every* sequence with 2 heads and 3 tails has that same probability. How many such sequences are there?

- $5!$ orders the five coins — but that's overkill, because it distinguishes identical coins.
- Divide by $2!$ to remove swaps of the two heads.
- Divide by $3!$ to remove swaps of the three tails.

$$
\binom{5}{2} = \frac{5!}{2!\,3!} = 10
$$

In general, $\binom{n}{k}$ ("**n choose k**") counts the ways to place $k$ heads among $n$ tosses.

**Symmetry**: $\binom{n}{k} = \binom{n}{n-k}$, because choosing $k$ heads is the same as choosing $n-k$ tails — this is why the fair-coin PMF is symmetrical.

### The binomial PMF

Let the probability of heads be $p$ and let $X$ = number of heads in $n$ tosses. One particular order with $x$ heads has probability $p^x (1-p)^{n-x}$; multiplying by the number of orders:

$$
p(x) = \binom{n}{x} p^x (1-p)^{n-x}, \qquad x = 0, 1, \dots, n
$$

We write $X \sim \text{Binomial}(n, p)$ — the tilde reads "follows the distribution." The **parameters** are $n$ (number of tosses) and $p$ (probability of heads).

### A die is a biased coin

Throw a die 5 times and count the 1s. Each throw is "1 or not-1" — heads with $p = \frac{1}{6}$, tails with $\frac{5}{6}$. So the count of 1s is $\text{Binomial}(5, \frac{1}{6})$: same model, different parameter.


## 3. Worked Examples

### Example 1: Two heads in five fair flips

$$
P(X = 2) = \binom{5}{2} \left(\tfrac{1}{2}\right)^2 \left(\tfrac{1}{2}\right)^3 = 10 \cdot \tfrac{1}{32} = \tfrac{10}{32} \approx 0.3125
$$

— exactly the 10/32 from the previous notebook, now computed instead of counted.

### Example 2: A biased coin, $p = 0.3$

With $X \sim \text{Binomial}(5, 0.3)$, small numbers of heads become likelier and the PMF loses its symmetry — the peak shifts left.

### Example 3: Ones in ten dice throws

Throw a die 10 times and record the number of 1s: $X \sim \text{Binomial}(n = 10,\ p = \frac{1}{6} \approx 0.1667)$.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb, factorial

rng = np.random.default_rng(seed=42)


### The binomial coefficient


In [ ]:
print("5!/(2! 3!) =", factorial(5) // (factorial(2) * factorial(3)))
print("comb(5, 2) =", comb(5, 2))
print("Symmetry: comb(10, 3) =", comb(10, 3), " comb(10, 7) =", comb(10, 7))


### The PMF from scratch


In [ ]:
def binomial_pmf(x, n, p):
    return comb(n, x) * p ** x * (1 - p) ** (n - x)

print(f"P(X=2), n=5, p=0.5:   {binomial_pmf(2, 5, 0.5):.4f}  (= 10/32)")
print(f"P(X=3 ones), n=5 dice: {binomial_pmf(3, 5, 1/6):.4f}")

pmf = [binomial_pmf(x, 5, 0.5) for x in range(6)]
print(f"Full PMF, Binomial(5, 0.5): {[f'{v:.4f}' for v in pmf]}")
print(f"Sums to: {sum(pmf):.4f}")


### Simulation check


In [ ]:
n_runs = 200_000
heads = (rng.random((n_runs, 5)) < 0.5).sum(axis=1)
ones = (rng.integers(1, 7, size=(n_runs, 10)) == 1).sum(axis=1)

print("x   sim Binomial(5,0.5)   exact")
for x in range(6):
    print(f"{x}        {(heads == x).mean():.4f}         {binomial_pmf(x, 5, 0.5):.4f}")

print(f"\nP(3 ones in 10 throws): sim {(ones == 3).mean():.4f}, "
      f"exact {binomial_pmf(3, 10, 1/6):.4f}")


## 5. Visualizations

### Fair vs. biased: the effect of p

Same $n = 5$; changing $p$ shifts and skews the distribution. $p = 0.5$ is symmetric (the $\binom{n}{k} = \binom{n}{n-k}$ symmetry); $p = 0.3$ leans left; $p = 1/6$ (the die) leans further.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for ax, p, title in [(axes[0], 0.5, "fair coin: p = 0.5"),
                     (axes[1], 0.3, "biased coin: p = 0.3"),
                     (axes[2], 1/6, "die counting 1s: p = 1/6")]:
    pmf = [binomial_pmf(x, 5, p) for x in range(6)]
    ax.bar(range(6), pmf, color="#4e79a7", edgecolor="white")
    ax.set_title(f"Binomial(5, {p:.3g})\n{title}")
    ax.set_xlabel("x (number of successes)")
    ax.grid(True, alpha=0.3, axis="y")

axes[0].set_ylabel("p(x)")
plt.tight_layout()
plt.show()


### Growing n: Binomial(10, 1/6)

The video's final example — ten dice throws, counting 1s.


In [ ]:
pmf10 = [binomial_pmf(x, 10, 1/6) for x in range(11)]

plt.figure(figsize=(9, 5))
plt.bar(range(11), pmf10, color="#f28e2b", edgecolor="white")
plt.title("Binomial(n=10, p=1/6): number of 1s in ten dice throws")
plt.xlabel("x (number of 1s)")
plt.ylabel("p(x)")
plt.xticks(range(11))
plt.grid(True, alpha=0.3, axis="y")
plt.show()


### The coefficients are Pascal's triangle

Each PMF numerator row for a fair coin is a row of Pascal's triangle — the pattern spotted in the previous notebook's exercise, now with a formula behind it.


In [ ]:
for n in range(1, 7):
    row = [comb(n, k) for k in range(n + 1)]
    print(f"n = {n}:  {row}")


### The whole story on one page

Everything above in a single sheet: counting orderings with the binomial coefficient, Binomial(5, 0.5) and Binomial(10, 1/6), the effect of p, Pascal's triangle, and the PMF formula.

<img src="img/03_binomial_distribution.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

1. Compute $\binom{6}{2}$ by the factorial formula and check with `comb`.
2. What are the parameters $n$ and $p$ if you flip a fair coin 20 times and count heads?

### Intermediate

3. What is the probability of getting **at most one** 1 in five dice throws? (Sum two PMF values.)
4. A production line has a 2% defect rate. In a shipment of 50 products, what is the probability of exactly 0 defects? Of exactly 2? Name the distribution and its parameters.

### Advanced

5. For $X \sim \text{Binomial}(10, 0.5)$, verify numerically that the PMF is symmetric ($p(x) = p(10-x)$ for all $x$), and show the symmetry breaks for $p = 0.3$.


## 7. Solutions


In [ ]:
# Exercise 1
print("1.  6!/(2!4!) =", factorial(6) // (factorial(2) * factorial(4)),
      "= comb(6,2) =", comb(6, 2))

# Exercise 2
print("2.  X ~ Binomial(n=20, p=0.5).")

# Exercise 3
p_at_most_1 = binomial_pmf(0, 5, 1/6) + binomial_pmf(1, 5, 1/6)
print(f"3.  P(X<=1) = p(0)+p(1) = {p_at_most_1:.4f}")

# Exercise 4
print(f"4.  X ~ Binomial(50, 0.02): p(0) = {binomial_pmf(0, 50, 0.02):.4f}, "
      f"p(2) = {binomial_pmf(2, 50, 0.02):.4f}")

# Exercise 5
sym = all(np.isclose(binomial_pmf(x, 10, 0.5), binomial_pmf(10 - x, 10, 0.5))
          for x in range(11))
asym = [round(binomial_pmf(x, 10, 0.3) - binomial_pmf(10 - x, 10, 0.3), 4)
        for x in range(4)]
print(f"5.  p=0.5 symmetric: {sym}; p=0.3 differences p(x)-p(10-x): {asym}")


## 8. Conclusion

- The **binomial coefficient** $\binom{n}{k} = \frac{n!}{k!(n-k)!}$ counts orderings of $k$ successes among $n$ trials — no listing required.
- $X \sim \text{Binomial}(n, p)$ has PMF $p(x) = \binom{n}{x} p^x (1-p)^{n-x}$; its two **parameters** are the number of trials and the success probability.
- One model covers many experiments: fair coins ($p = 0.5$), biased coins, dice counting a face ($p = \frac{1}{6}$), defect counts — anything that is a fixed number of independent yes/no trials.
- $p = 0.5$ gives a symmetric PMF (via $\binom{n}{k} = \binom{n}{n-k}$); other values of $p$ skew it.
